# YOLO: подготовка датасета и обучение

Этот ноутбук посвящён кастомному датасету: как аккуратно разложить файлы, как проверить структуру, как создать `dataset.yaml` и как запустить обучение с минимальным количеством команд.


## Формат датасета YOLO

Обычно структура такая:

```
dataset/
    images/
        train/
        val/
    labels/
        train/
        val/
```

Для каждого изображения должен быть текстовый файл с тем же именем и расширением `.txt`.
Внутри каждой строки: `class_id x_center y_center width height` в нормализованных координатах.


In [ ]:

from pathlib import Path
import yaml
import random
import os

import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO


In [ ]:

# Основные настройки датасета.
# При реальной работе поменяйте ROOT_DATASET_DIR на свою папку.
ROOT_DATASET_DIR = Path("dataset")
TRAIN_IMAGES_DIR = ROOT_DATASET_DIR / "images" / "train"
VAL_IMAGES_DIR = ROOT_DATASET_DIR / "images" / "val"
TRAIN_LABELS_DIR = ROOT_DATASET_DIR / "labels" / "train"
VAL_LABELS_DIR = ROOT_DATASET_DIR / "labels" / "val"

# Имена классов. Важно соблюдать порядок: индекс в списке = class_id.
CLASS_NAMES = ["class_0"]

# Имя файла конфигурации датасета.
DATASET_YAML_PATH = ROOT_DATASET_DIR / "dataset.yaml"


## Создание папок

Эта ячейка удобна для пустой заготовки. Она создаст необходимую структуру, если её ещё нет.


In [ ]:

for p in [TRAIN_IMAGES_DIR, VAL_IMAGES_DIR, TRAIN_LABELS_DIR, VAL_LABELS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Структура директорий готова")


## Проверка согласованности изображений и разметки

Очень частая проблема — есть изображения без `.txt` или наоборот. Ниже базовая проверка, которую полезно запускать перед обучением.


In [ ]:

def find_pairs(images_dir: Path, labels_dir: Path):
    image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    image_stems = {p.stem for p in images_dir.iterdir() if p.suffix.lower() in image_exts} if images_dir.exists() else set()
    label_stems = {p.stem for p in labels_dir.iterdir() if p.suffix.lower() == ".txt"} if labels_dir.exists() else set()

    missing_labels = sorted(image_stems - label_stems)
    missing_images = sorted(label_stems - image_stems)
    matched = sorted(image_stems & label_stems)
    return matched, missing_labels, missing_images

for split_name, img_dir, lbl_dir in [
    ("train", TRAIN_IMAGES_DIR, TRAIN_LABELS_DIR),
    ("val", VAL_IMAGES_DIR, VAL_LABELS_DIR),
]:
    matched, missing_labels, missing_images = find_pairs(img_dir, lbl_dir)
    print(f"=== {split_name} ===")
    print("matched:", len(matched))
    print("images without labels:", len(missing_labels))
    print("labels without images:", len(missing_images))
    if missing_labels[:5]:
        print("Примеры изображений без разметки:", missing_labels[:5])
    if missing_images[:5]:
        print("Примеры разметки без изображений:", missing_images[:5])
    print()


## Генерация `dataset.yaml`

YOLO использует YAML-файл с путями к данным и списком классов. Ниже автоматическая генерация такого файла.


In [ ]:

dataset_yaml = {
    "path": str(ROOT_DATASET_DIR.resolve()),
    "train": "images/train",
    "val": "images/val",
    "names": {i: name for i, name in enumerate(CLASS_NAMES)},
}

with open(DATASET_YAML_PATH, "w", encoding="utf-8") as f:
    yaml.safe_dump(dataset_yaml, f, allow_unicode=True, sort_keys=False)

print("Файл создан:", DATASET_YAML_PATH)
print(DATASET_YAML_PATH.read_text(encoding="utf-8"))


## Визуальная проверка разметки

Перед запуском обучения полезно глазами убедиться, что bounding boxes реально попадают в объекты, а классы корректны.


In [ ]:

def read_yolo_labels(label_path: Path):
    rows = []
    if not label_path.exists():
        return rows
    for line in label_path.read_text(encoding="utf-8").splitlines():
        parts = line.strip().split()
        if len(parts) != 5:
            continue
        cls_id, xc, yc, w, h = parts
        rows.append((int(cls_id), float(xc), float(yc), float(w), float(h)))
    return rows

def draw_boxes(image_path: Path, label_path: Path, class_names):
    img = Image.open(image_path).convert("RGB")
    w_img, h_img = img.size
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.imshow(img)

    for cls_id, xc, yc, bw, bh in read_yolo_labels(label_path):
        x1 = (xc - bw / 2) * w_img
        y1 = (yc - bh / 2) * h_img
        rect_w = bw * w_img
        rect_h = bh * h_img
        rect = plt.Rectangle((x1, y1), rect_w, rect_h, fill=False)
        ax.add_patch(rect)
        ax.text(x1, y1, class_names[cls_id], fontsize=10)

    ax.axis("off")
    plt.show()


In [ ]:

# Попробуйте выбрать случайный пример из train и посмотреть разметку.
train_images = sorted([p for p in TRAIN_IMAGES_DIR.glob("*") if p.is_file()])

if train_images:
    sample_image = random.choice(train_images)
    sample_label = TRAIN_LABELS_DIR / f"{sample_image.stem}.txt"
    print("Проверяем файл:", sample_image.name)
    draw_boxes(sample_image, sample_label, CLASS_NAMES)
else:
    print("В train пока нет изображений")


## Обучение модели

Для быстрого старта обычно достаточно `yolov8n.pt`. Ниже базовый запуск обучения через Python API.


In [ ]:

PRETRAINED_WEIGHTS = "yolov8n.pt"
EPOCHS = 20
IMAGE_SIZE = 640
BATCH = 16
PROJECT_NAME = "runs_train"
RUN_NAME = "exp_yolo"


In [ ]:

# Обратите внимание:
# - если данных мало, лучше начать с небольшого числа эпох и быстрой модели
# - для CPU можно уменьшить batch
# - imgsz влияет и на качество, и на скорость

# Эта ячейка запускает реальное обучение.
# Перед выполнением убедитесь, что датасет подготовлен.

# model = YOLO(PRETRAINED_WEIGHTS)
# train_results = model.train(
#     data=str(DATASET_YAML_PATH),
#     epochs=EPOCHS,
#     imgsz=IMAGE_SIZE,
#     batch=BATCH,
#     project=PROJECT_NAME,
#     name=RUN_NAME,
#     exist_ok=True,
# )


## Эквивалент в одну команду из терминала

Иногда удобнее запускать обучение без ноутбука. Тогда можно использовать CLI: 

```bash
yolo detect train data=dataset/dataset.yaml model=yolov8n.pt epochs=20 imgsz=640
```


## Что полезно менять на практике

- `model` — размер и скорость модели
- `epochs` — длительность обучения
- `imgsz` — компромисс между качеством и скоростью
- `batch` — зависит от памяти
- `patience` — ранняя остановка
- `freeze` — если нужно заморозить часть слоёв
- `device` — CPU или GPU


## Что делать дальше

После обучения обычно переходят к валидации и разбору ошибок. Это вынесено в третий ноутбук.
